# Earth2Studio Interop Prototype

This notebook demonstrates how `seisfetch` data feeds into
[NVIDIA Earth2Studio](https://nvidia.github.io/earth2studio/) via the
adapter classes in `seisfetch.earth2`.

Earth2Studio defines protocol-based interfaces:
- **`DataSource`**: `__call__(time, variable) → xr.DataArray` — for gridded data
- **`DataFrameSource`**: `__call__(time, variable, fields) → pd.DataFrame` — for sparse sensor data

Seismic waveforms are sparse sensor time-series, so both protocols can be useful:
- `SeismicDataSource` → waveform arrays with dims `[time, variable, sample]`
- `SeismicDataFrameSource` → per-station summary rows (RMS, max amplitude, lat/lon)

In [ ]:
from datetime import datetime

# seisfetch core
from seisfetch import bundle_to_xarray, parse_mseed
from seisfetch.earth2 import SeismicDataFrameSource, SeismicDataSource, bundle_to_earth2

In [ ]:
import sys

import pymseed

print(sys.executable)
print(pymseed.__version__)

## 1. Fetch seismic data via S3 (SCEDC)

Download one day of broadband data from the Southern California Earthquake
Data Center.

In [ ]:
from seisfetch import SeisfetchClient

client = SeisfetchClient(backend="s3_open", datacenter="scedc")
raw = client.get_raw(
    network="CI",
    station="PASC",
    location="00",
    channel="BHZ",
    starttime="2024-01-15T00:00:00",
    endtime="2024-01-15T01:00:00",
)
bundle = parse_mseed(raw)
print(f"Parsed {len(bundle)} traces: {bundle.ids}")

## 2. Seisfetch → xarray Dataset (existing)

In [ ]:
ds = bundle_to_xarray(bundle)
ds

## 3. SeismicDataSource — Earth2Studio DataSource protocol

Wraps the bundle into a callable that returns
`xr.DataArray(dims=[time, variable, sample])` — the shape Earth2Studio expects.

In [ ]:
source = SeismicDataSource(bundle)

# Query it like Earth2Studio would
var_names = list(source._var_names)
print(f"Available variables: {var_names}")

da = source(datetime(2024, 1, 15), var_names)
print(f"\nResult shape: {da.shape}  dims: {da.dims}")
da

## 4. SeismicDataFrameSource — Earth2Studio DataFrameSource protocol

Produces a `pd.DataFrame` with one row per station-channel,
including RMS/max amplitude and optional lat/lon.
This maps to Earth2Studio's sparse-sensor interface.

In [ ]:
station_coords = {
    "CI.PASC": (34.1715, -118.1870),
}

df_source = SeismicDataFrameSource(bundle, station_coords=station_coords)
df = df_source(datetime(2024, 1, 15), var_names)
df

## 5. Convenience: `bundle_to_earth2`

In [ ]:
src = bundle_to_earth2(bundle)
type(src)

## 6. Protocol verification

If `earth2studio` is installed, we can verify our adapters pass the
runtime-checkable protocol.

In [ ]:
try:
    from earth2studio.data.base import DataFrameSource, DataSource

    print(f"SeismicDataSource  is DataSource:      {isinstance(source, DataSource)}")
    print(
        f"SeismicDataFrameSource is DataFrameSource: {isinstance(df_source, DataFrameSource)}"
    )
except ImportError:
    print("earth2studio not installed — skipping protocol check.")
    print("Install with: pip install 'seisfetch[earth2]'")

## Next Steps

With these adapters in place, seismic data from seisfetch can be plugged
directly into Earth2Studio workflows:

```python
from earth2studio.run import deterministic as run
from earth2studio.models.dx import SomeModel
from earth2studio.io import ZarrBackend

# seisfetch data as Earth2Studio source
data = SeismicDataSource(bundle)
io = ZarrBackend("outputs/seismic_inference.zarr")
# ... feed into an Earth2Studio pipeline
```

Potential use cases:
1. **Site-effect prediction** — feed station waveforms into an AI model
2. **Ground-motion estimation** — use sparse sensor DataFrame as input
3. **Combined weather-seismic** — merge seismic data with weather DataSources